In [1]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas_datareader import data
from sklearn.metrics import mean_squared_error
from pmdarima import auto_arima
from statsmodels.tsa.arima_model import ARIMA
from prophet import Prophet

import warnings
warnings.filterwarnings("ignore")

Importing plotly failed. Interactive plots will not work.


In [9]:
def predict_price(ticker, ml_model):
    
    start_date = '2019-01-01'

    today = datetime.date.today()
    yesterday = today - datetime.timedelta(days=1)
    end_date = yesterday

    panel_data = data.DataReader([ticker], 'yahoo', start_date, end_date)
    
    # ml model: auto arima, arima, prohphet, linear regression, LSTM
    if ml_model == 'auto arima':
        return ml_auto_arima(ticker, panel_data)
    elif ml_model == 'arima':
        return ml_arima(ticker, panel_data)
    elif ml_model == 'prophet':
        return ml_prophet(ticker, panel_data)

In [10]:
#test
predict_price('AAPL', 'auto arima')

{'today_price': 155.7742945363473,
 'yesterday_price': 151.27999877929688,
 'real_vs_prediction':             Prediction        AAPL
 Date                              
 2021-04-15  132.541899  134.500000
 2021-04-16  132.655632  134.160004
 2021-04-19  132.823810  134.839996
 2021-04-20  132.984543  133.110001
 2021-04-21  133.146294  133.500000
 ...                ...         ...
 2021-11-01  154.966151  148.960007
 2021-11-02  155.127780  150.020004
 2021-11-03  155.289408  151.490005
 2021-11-04  155.451037  150.960007
 2021-11-05         NaN  151.279999
 
 [144 rows x 2 columns],
 'rmse': 45.46257774903891}

tickers = ['AAPL']

start_date = '2019-01-01'

today = datetime.date.today()
yesterday = today - datetime.timedelta(days=1)
end_date = yesterday

'''User pandas_reader.data.DataReader to load data'''
panel_data = data.DataReader(tickers, 'yahoo', start_date, end_date)


'''df = panel_data['Close']'''
df = panel_data
df.head()

# Auto ARIMA

In [5]:
from pmdarima import auto_arima

def ml_auto_arima(ticker, df):
    close = df['Close']
    
    #split data (train:test = 0.8:0.2)
    train_ratio = 0.8
    train_size = int(df.shape[0] * train_ratio)
    test_size = df.shape[0] - train_size

    train = close.head(train_size)
    test = close.tail(test_size)
    
    # training the model
    model = auto_arima(train, start_p=1, start_q=1,max_p=6, max_q=3, m=12, seasonal=True, error_action='ignore',suppress_warnings=True)
    model.fit(train)
    
    # prediction
    forecast = model.predict(n_periods=test_size+1)
    today_prediction = forecast[-1]
    forecast = pd.DataFrame(forecast[:-1],index = test.index,columns=['Prediction'])
    
    # combine test and forecast dataframes
    result = pd.concat([forecast[:-1], test], axis = 1)
    # result.plot()

    # calculating rmse 
    MSE_error = mean_squared_error(test, forecast)
    
    # output 
    return {'today_price': today_prediction, 'yesterday_price': test[ticker][-1], 'real_vs_prediction': result, 'rmse': MSE_error}

In [ ]:
#test
ml_auto_arima('AAPL', panel_data)

In [ ]:
# 5 days future price prediction
from pmdarima import auto_arima

close = df['Close']
# splitting the data
train_ratio = 0.8
train_size = int(df.shape[0] * train_ratio)
test_size = df.shape[0] - train_size

train = close.head(train_size)
test = close.tail(test_size)

# train.plot()
# test.plot()

model = auto_arima(train, start_p=1, start_q=1,max_p=6, max_q=3, m=12, seasonal=True, error_action='ignore',suppress_warnings=True)
model.fit(train)

forecast = model.predict(n_periods=test_size+5)
today_prediction = forecast[-5:]
forecast = pd.DataFrame(forecast[:-5],index = test.index,columns=['Prediction'])
# forecast = pd.DataFrame(forecast,columns=['Prediction'])

result = pd.concat([forecast, test], axis = 1)
result.plot()

# calculating rmse
MSE_error = mean_squared_error(test, forecast)
print('Testing Mean Squared Error is {}'.format(MSE_error))

# ARIMA

In [6]:
def ml_arima(ticker, df):
    
    # preprocessing data
    train_data, test_data = df[0:int(len(df)*0.8)], df[int(len(df)*0.8):]
    training_data = train_data['Close'].values
    test_data = test_data['Close'].values
    history = [x for x in training_data]
    model_predictions = []
    N_test_observations = len(test_data)
    
    # train model
    for time_point in range(N_test_observations+1):
        try:
            model = ARIMA(history, order=(4,1,0))
            model_fit = model.fit(disp=0)
            output = model_fit.forecast()
            yhat = output[0]
            true_test_value = test_data[time_point]
            history.append(true_test_value)
            model_predictions.append(yhat)
        except:
            today_prediction = yhat
    
    model_predictions = pd.DataFrame(model_predictions,index = test.index,columns=['Prediction'])
        
    # combine test and forecast dataframes
    result = pd.concat([model_predictions[:-1], test], axis = 1)
    
    # calculating rmse
    MSE_error = mean_squared_error(test_data, model_predictions)
    
    # output
    return {'today_price': today_prediction[0], 'yesterday_price': test_data[-1], 'real_vs_prediction': result, 'rmse': MSE_error}

In [ ]:
#test
ml_arima('AAPL', panel_data)

In [ ]:
# future 5 days prediction
from statsmodels.tsa.arima_model import ARIMA

train_data, test_data = df[0:int(len(df)*0.7)], df[int(len(df)*0.7):]
training_data = train_data['Close'].values
test_data = test_data['Close'].values
history = [x for x in training_data]
model_predictions = []
future = []
N_test_observations = len(test_data)
for time_point in range(N_test_observations+5):
    try:
        model = ARIMA(history, order=(4,1,0))
        model_fit = model.fit(disp=0)
        output = model_fit.forecast()
        yhat = output[0]
        true_test_value = test_data[time_point]
        history.append(true_test_value)
        model_predictions.append(yhat)
    except:
        future.append(yhat)
        history.append(yhat)
        model_predictions.append(yhat)
#         today_stock_price = yhat
MSE_error = mean_squared_error(test_data, model_predictions[:-5])
print('Testing Mean Squared Error is {}'.format(MSE_error))

In [ ]:
test_set_range = df[int(len(df)*0.7):].index
plt.plot(test_set_range, model_predictions, color='blue', marker='o', linestyle='dashed',label='Predicted Price')
plt.plot(test_set_range, test_data, color='red', label='Actual Price')
plt.title('AAPL Prices Prediction')
plt.xlabel('Date')
plt.ylabel('Prices')
# plt.xticks(np.arange(881,1259,50), df.Date[881:1259:50])
plt.legend()
plt.show()


#future predictions
print("Today's price prediction: ", yhat[0])

# Prophet

In [8]:
def ml_prophet(ticker, df):
    close = df['Close']

    #preparing data
    close.reset_index(inplace = True)
    close = close[['Date', 'AAPL']]
    close.rename(columns={'AAPL': 'y', 'Date':'ds'}, inplace = True)

    #splitting the data
    train_ratio = 0.8
    train_size = int(df.shape[0] * train_ratio)
    test_size = df.shape[0] - train_size

    train = close.head(train_size)
    test = close.tail(test_size)

    #fit the model
    model = Prophet(daily_seasonality=True)
    model.fit(train)

    #predictions
    close_prices = model.make_future_dataframe(periods=len(test)+1)
    forecast = model.predict(close_prices)
    today_prediction = forecast['yhat'].to_list()[-1]
    
    # combine test and forecast dataframes
    forecast = pd.DataFrame(forecast,index = test.index,columns=['yhat'])
    result = pd.concat([forecast, test], axis = 1)
    
    #rmse
    forecast_valid = forecast['yhat'][-test_size:]
    MSE_error = mean_squared_error(test['y'], forecast_valid)
    
    # renaming columns in result
    result.rename(columns = {'yhat':'Prediction', 'y': ticker}, inplace = True)
    result.drop(columns = ['ds'], inplace = True)
    
    # output
    return {'today_price': today_prediction, 'yesterday_price': test['y'].to_list()[-1], 'real_vs_prediction': result, 'rmse': MSE_error}

In [ ]:
#test
# ml_prophet('AAPL', panel_data)

In [ ]:
# suppose to predict 5 days (work in progress)
from prophet import Prophet

close = df['Close']

#preparing data
close.reset_index(inplace = True)
close = close[['Date', 'AAPL']]
close.rename(columns={'AAPL': 'y', 'Date':'ds'}, inplace = True)

#splitting the data
train_ratio = 0.8
train_size = int(df.shape[0] * train_ratio)
test_size = df.shape[0] - train_size

train = close.head(train_size)
test = close.tail(test_size)

#fit the model
model = Prophet(daily_seasonality=True)
model.fit(train)

#predictions
close_prices = model.make_future_dataframe(periods=len(test))
forecast = model.predict(close_prices)

#rmse
forecast_valid = forecast['yhat'][-test_size:]
MSE_error = mean_squared_error(test['y'], forecast_valid)
print('Testing Mean Squared Error is {}'.format(MSE_error))

#plot
test['Predictions'] = 0
test['Predictions'] = forecast_valid.values

# plt.plot(train['y'])
plt.plot(test['y'], color = 'blue', label = 'Actual Price')
plt.plot(test['Predictions'], color = 'red', label = 'Predicted Price')
plt.title('AAPL Prices Prediction')
plt.xlabel('Date')
plt.ylabel('Prices')
plt.legend()
plt.show()

# LSTM

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM

In [ ]:
close = df['Close']

#preparing data
close.reset_index(inplace = True)
close = close[['Date', 'AAPL']]

#splitting the data
train_ratio = 0.8
train_size = int(df.shape[0] * train_ratio)
test_size = df.shape[0] - train_size

train = close.head(train_size)
test = close.tail(test_size)

In [ ]:
# from sklearn.linear_model import LinearRegression

# model = LinearRegression()
# model.fit(training_data)